In [1]:
pip install wfdb torch transformers neurokit2 tslearn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import wfdb
import numpy as np
from sklearn.model_selection import train_test_split
import torch
import pandas as pd
import sys
import json
import matplotlib.pyplot as plt
from collections import defaultdict
from tslearn.piecewise import SymbolicAggregateApproximation, OneD_SymbolicAggregateApproximation


#from transformers import PreTrainedTokenizerFast, BigBirdForSequenceClassification, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
import importlib
import ECGSignalPreprocessor_new
import compat_tokenizer_1d
import os
importlib.reload(ECGSignalPreprocessor_new)
importlib.reload(compat_tokenizer_1d)

#IMPORT ECGSIGNALPREPROCESSOR AND COMPAT_TOKENIZER_1D MANUALLY. CONNECT TO DRIVE FOR THE ECG FILES!!

def load_ecg_signals(filepaths):
    all_data = []
    for path in filepaths:
        record = wfdb.rdrecord(path)
        signal = np.transpose(record.p_signal[:,0:12])
        x = np.linspace(0, 10, record.sig_len)

        processor = ECGSignalPreprocessor_new.ECGSignalProcessor(
            signal, 12, record.sig_len, record.sig_name, x)
        processor.detect_peaks_and_dips()
        processor.apply_bandpass_filter()
        processor.apply_notch_filter(processor.filtered_signal)
        processor.apply_savgol_filter(processor.filtered_signal)
        processor.apply_detrend(processor.smoothed)
        trimmed_signal = processor.trim_signal(processor.detrend)
        all_data.append(trimmed_signal)
    return all_data


# Padding
def pad_signal(signal, target_length):
    signal = np.asarray(signal)
    return np.array([
        np.pad(lead, (0, max(0, target_length - len(lead))), mode='wrap')[:target_length]
        for lead in signal])

#Labels
df = pd.read_csv("drive/MyDrive/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/ptbxl_database.csv", sep=',')
diagnosis_df = df['report']
labels = diagnosis_df.to_numpy()


#Filepaths
filepaths = []
start_line = 43605

with open("drive/MyDrive/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/SHA256SUMS.txt", "r") as f:
    for i, line in enumerate(f):
        if i < start_line:
            continue
        parts = line.strip().split()
        if len(parts) != 2:
            continue
        filepath = parts[1]
        root = "drive/MyDrive/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
        if filepath.endswith(".hea"):
            base = filepath[:-4]  # remove ".hea"
            full_paths = os.path.join(root, base)
            filepaths.append(full_paths)


all_data = load_ecg_signals(filepaths)
padded_signals = [pad_signal(sig, target_length = 5000) for sig in all_data]

#Concatenate all the signals in one to get a unique fit
concatenated_signals = np.concatenate([sig.flatten() for sig in padded_signals])
concatenated_signals = concatenated_signals.reshape(1, -1)
print(len(concatenated_signals[0]))
sax = OneD_SymbolicAggregateApproximation(n_segments=100, alphabet_size_avg=300, alphabet_size_slope=300)
n_segments = sax.n_segments
alphabet_size = sax.alphabet_size_avg
sax_list = sax.fit(concatenated_signals)



#Use compat_tokenizer_1d
vocab_file={}
tok_model = compat_tokenizer_1d.Word_Tokenizer(filepaths, n_segments, alphabet_size, vocab_file, SAX_List = sax_list)
toks_sax, toks_sax_inv = tok_model.tokenize_sax(padded_signals) #Apply SAX
reshaped_toks = tok_model.reshape_tokens(toks_sax)  #Reshape the token arrays
toks2sentence = tok_model.token_to_string(reshaped_toks)  #Concatenate the leads and make sentenece

toks2sentence_list = list(toks2sentence)
output_filename = "toks2sentence.json"

with open(output_filename, "w") as f:
    json.dump(toks2sentence_list, f, indent=2)

In [ ]:
#Label handling
count_dict = defaultdict(int)
for label in labels:
    count_dict[label.strip()]+=1

print(count_dict)

#Remove labels occuring less than 15 times
valid_labels = {label for label, count in count_dict.items() if count >= 15}
print(valid_labels)

filtered_samples = []
filtered_labels = []

for i in range(len(labels)):
    label = labels[i].strip()
    if label in valid_labels:
        filtered_samples.append(toks2sentence[i])
        filtered_labels.append(label)

#Label to id
unique_labels = sorted(set(filtered_labels))
label2id = {label: idx for idx, label in enumerate(unique_labels)}
label_id = [label2id[label.strip()] for label in filtered_labels]

num_labels = len(unique_labels)

for label in sorted(unique_labels):
    print(f"Label: {label} | ID: {label2id[label]} | Frequency: {filtered_labels.count(label)}")
print(type(filtered_samples[0]))
print(f"Total dataset size: {len(filtered_samples)}")

In [ ]:
#Split train and eval
train_texts, val_texts, train_labels, val_labels = train_test_split(
    filtered_samples, filtered_labels, test_size=0.2, random_state=42)

train_dataset = tok_model.create_ecg_dataset(train_texts, train_labels)
val_dataset = tok_model.create_ecg_dataset(val_texts, val_labels)

vocab_size = tok_model.vocab_size()

vocab = tok_model.get_vocab()

decoded_texts = []
inverse_signals = []

for i, text in enumerate(val_texts):

    sax_token = tok_model._tokenize(text)
    #print("SAXtoken:" + str(sax_token))

    token_to_id = [tok_model._convert_token_to_id(tok) for tok in sax_token]
    #print("token to id:" , token_to_id)

    id_to_token = [tok_model._convert_id_to_token(tok_id) for tok_id in token_to_id]
    #print("id to token:" , id_to_token)

    decode , inverse = tok_model._decode(token_to_id, skip_special_tokens = True)
    #print("decoded:" + decode)
    #print(len(inverse)) --> 5000
    decoded_texts.append(decode)
    inverse_signals.append(inverse)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4)

for batch in train_loader:
    input_ids = batch['input_ids']
    attention_mask = batch['attention_mask']
    labels = batch['labels']
    print("Input IDs:", input_ids)
    print("Labels:", labels)
    break

In [ ]:
# Load BigBird model and train
from transformers import BigBirdForSequenceClassification

model = BigBirdForSequenceClassification.from_pretrained("google/bigbird-roberta-base", num_labels=num_labels)
model.resize_token_embeddings(vocab_size)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)
model.train()

for epoch in range(3):
    total_loss = 0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()
     print(f"Epoch {epoch+1} - Loss: {total_loss:.4f}")